In [1]:
import os, sys
sys.path.append("E:\\Dai hoc\\2526I\\dacn\\flow-matching\\pep2prob")
os.environ["TORCH_HOME"] = "E:\\Dai hoc\\2526I\\dacn\\flow-matching\\.cache"
# os.environ["PYTHONPATH"] = "E:\\Dai hoc\\2526I\\dacn\\flow-matching\\pep2prob"

In [2]:
from time import time
import torch
from torch.nn.functional import mse_loss
import esm
from flow_matching.path import CondOTProbPath
from flow_matching.utils import ModelWrapper
from flow_matching.solver import ODESolver
# from flow_matching.loss import 

In [3]:
from my_model.mlp.mlp import FlowMLP
from p2p_dataset import Pep2ProbDataset


In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [39]:
device.type
torch.set_default_device('cuda')

In [4]:
peptide_embedding_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
peptide_embedding_model.eval()
batch_converter = alphabet.get_batch_converter()
pep2prob_ds = Pep2ProbDataset(data_dir="data", max_length_input=7)

Download complete.
Dataset and data split downloaded successfully.


In [7]:
pep2prob_ds.train[0]

(array(['AAANQMR', 2], dtype=object),
 array([ 1.      ,  0.      ,  1.      ,  0.885856,  0.312655,  0.039702,
         0.007444, -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      ,  0.      ,  0.      ,
         0.004963,  0.007444,  0.      ,  0.042184, -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
        -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,


In [12]:
# count how frequency of pep seq in training set
from collections import defaultdict
pep_freq = defaultdict(int)
for pep_data in pep2prob_ds.train:
    pep_seq, _, __, ___ = pep_data
    # print(pep_seq[0])
    pep_freq[pep_seq[0]] += 1

pep_freq

defaultdict(int,
            {'AAANQMR': 1,
             'AAATPAK': 1,
             'AAEVYTR': 1,
             'AAFFAER': 1,
             'AAFGQVR': 1,
             'AAFNSGK': 1,
             'AAFTAGR': 1,
             'AAHFVFR': 2,
             'AAIDSYR': 1,
             'AALNAVR': 1,
             'AAMALEK': 1,
             'AAMAQWR': 1,
             'AAQANVR': 1,
             'AAQLGFK': 1,
             'AAQNISK': 1,
             'AAQVTFR': 1,
             'AASVPVK': 1,
             'AATMEVK': 1,
             'AATQFWR': 1,
             'AATVVAR': 1,
             'AAVDYQK': 1,
             'AAVEFNK': 1,
             'AAVFHEK': 1,
             'AAVGVKK': 1,
             'AAVHYDR': 1,
             'AAVPSIK': 1,
             'AAVTPGK': 1,
             'AAVVVSK': 1,
             'AAWEAGK': 1,
             'AAWEHMK': 1,
             'AAWNALR': 1,
             'AAYIYIR': 1,
             'AAYNLVK': 1,
             'AAYNLVR': 1,
             'AAYNQVK': 1,
             'ADAVWNK': 1,
           

In [48]:
# Only take peptides with length = 7
len(pep2prob_ds.train)


5452

In [42]:
def extract_ds_row(ds_row: tuple):
    (precursor, probs, prob_masks, info) = ds_row
    return {
        "peptide": precursor[0],
        "charge": precursor[1],
        "probs": probs,     
        "prob_masks": prob_masks,  
        "info": info
    }
    
def extract_batch(batch: list[tuple]):
    # print("Extracting batch of size:", len(batch))
    return [extract_ds_row(row) for row in batch]

def get_peptide_embs(peptides: list[str], device="cuda"):
    batch_labels, batch_strs, batch_tokens = batch_converter(
        [(f"p_{i}", pep) for i, pep in enumerate(peptides)]
    )
    
    batch_tokens = batch_tokens.to(device)
    
    with torch.no_grad():
        results = peptide_embedding_model(
            batch_tokens, 
            repr_layers=[12], 
            return_contacts=False
        )
    
    token_reps = results["representations"][12]
    
    # Efficiently calculate mean embeddings excluding special tokens
    peptide_embeddings = []
    for i, label in enumerate(batch_labels):
        seq_len = len(label)
        emb = token_reps[i, 1 : seq_len + 1].mean(dim=0)
        peptide_embeddings.append(emb)
        
    return torch.stack(peptide_embeddings)

def collate_fn(batch):
    peptides = [item["peptide"] for item in batch]
    probs = torch.stack([torch.as_tensor(item["probs"]) for item in batch]).to(device)
    prob_masks = torch.stack([torch.as_tensor(item["prob_masks"]) for item in batch]).to(device)
    charges = torch.tensor([item["charge"] for item in batch], dtype=torch.float32, device=device)
    peptide_embs = get_peptide_embs(peptides, device=device)
    infos = [item["info"] for item in batch]
    
    return {
        "peptide_embs": peptide_embs,
        "charges": charges,
        "probs": probs,
        "prob_masks": prob_masks,
        "infos": infos
    }

In [ ]:
pep2prob_ds.train = pep2prob_ds.train.slice(0, 2560)

[(array(['AAANQMR', 2], dtype=object),
  array([ 1.      ,  0.      ,  1.      ,  0.885856,  0.312655,  0.039702,
          0.007444, -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      ,  0.      ,  0.      ,
          0.004963,  0.007444,  0.      ,  0.042184, -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.      , -1.      ,
         -1.      , -1.      , -1.      , -1.      , -1.     

In [44]:
collate_fn(extract_batch(pep2prob_ds.train[1:3]))["peptide_embs"].shape

torch.Size([2, 480])

In [45]:
model = FlowMLP(x_dim=235, c_dim=peptide_embedding_model.embed_dim, time_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-10)
path = CondOTProbPath()

In [ ]:
## train flow matching model
epoch = 1000
batch_size = 256
start = time()
print(f"Start training at {start:.2f} seconds")
for _ in range(epoch):
    for j in range(0, len(pep2prob_ds.train), batch_size):
        optimizer.zero_grad()
        batch_data = collate_fn(extract_batch(pep2prob_ds.train[j : j + batch_size]))
        
        peptide_embs = batch_data["peptide_embs"]   # (batch_size, c_dim)
        x_1 = batch_data["probs"] # target probs  (batch_size, x_dim)
        x_0 = torch.randn_like(x_1) # initial probs (batch_size, x_dim)
        t = torch.rand(x_1.size(0)) 
        
        u_pred = model(x_0, t, peptide_embs)
        path_sample = path.sample(x_0, x_1, t)
        loss = mse_loss(u_pred, path_sample.dx_t, reduction='mean')
        loss.backward()
        optimizer.step()
        
        if j % (batch_size * 100) == 0:
            print(f"Batch {j // batch_size}, Loss: {loss.item():.6f}")
        break
    break
        
end = time()
print(f"Finished training at {end:.2f} seconds, total time: {end - start:.2f} seconds")
    
    
    


Start training at 1769776510.90 seconds
Batch 0, Loss: 1.942632
Batch 0, Loss: 1.927375
Batch 0, Loss: 1.933039
Batch 0, Loss: 1.935658
Batch 0, Loss: 1.931275
Batch 0, Loss: 1.934443
Batch 0, Loss: 1.924784
Batch 0, Loss: 1.928987
Batch 0, Loss: 1.926494
Batch 0, Loss: 1.929557
Batch 0, Loss: 1.934261
Batch 0, Loss: 1.912836
Batch 0, Loss: 1.932517
Batch 0, Loss: 1.936622
Batch 0, Loss: 1.934264
Batch 0, Loss: 1.953549
Batch 0, Loss: 1.939195
Batch 0, Loss: 1.937696
Batch 0, Loss: 1.946497
Batch 0, Loss: 1.923910
Batch 0, Loss: 1.936857
Batch 0, Loss: 1.925227


KeyboardInterrupt: 

In [47]:
wrapped_model = ModelWrapper(model)

In [ ]:
# sample from trained model
sample_pep = pep2prob_ds.test[0]


In [ ]:
# # run test on trained model
# pep2prob_ds.test = pep2prob_ds.test.slice(0, 256)
# start = time()
# print(f"Start testing at {start:.2f} seconds")
# with torch.no_grad():
    
# end = time()
# print(f"Finished testing at {end:.2f} seconds, total time: {end - start:.2f} seconds")
